# Rail Alliance Pilot Markets

2026-03-19

In [1]:
import geopandas as gpd
import intake
import pandas as pd
# from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
# from shared_utils import catalog_utils
# from update_vars import analysis_date

catalog = intake.open_catalog("./catalog.yml")

In [4]:
from calitp_data_analysis.geography_utils import CA_NAD83Albers_m, WGS84

In [2]:
def print_stats(gdf):
    print(f"CRS: {gdf.crs.to_epsg()}")
    print(f"{gdf.columns}")
    print(gdf.dtypes)
    print(f"# rows: {len(gdf)}")

## CapCorr

* Origins
    * rail-served Bay Area 2000m buffer
    * select urban core frequent bus routes 1000m buffer (Muni 49, 38/38R, AC 1T, 72/72M/72L, 51A/51B, VTA 22/522)
    * Martinez station area (also a destination)
* Destinations
    * CapCor stations NE of MTZ
    * SacRT Rail
    * Martinez station area (also an origin)
* Other Constraints
    * Gateway on Carquinez Strait Crossings (I80, I680, CapCor Rail Bridge)
    * Min Distance 5mi (exclude intra-Martinez)
* Primary Insights
    * Trips viable by CapCorr + local transit
* Gaps
    * Short trips within Bay Area
    * Martinez - Bay Area trips
    * Further local markets, park and ride trips

### Current Travel Info (Replica)

### Potential Elasticities (non-work)

* Assuming 10% of non-work auto trips shift to local transit + CapCor, ##### new daily riders

### CapCor Origins

In [6]:
gdf = catalog.ca_transit_stops.read().to_crs(CA_NAD83Albers_m)

In [7]:
gdf.routetypes.unique()

array(['2', '3', '0', '5', '0, 3', '4', '2, 3', '1'], dtype=object)

In [8]:
muni_select = gdf[(gdf.agency == 'City and County of San Francisco') & ((gdf.route_ids_served.str.contains('49')) | (gdf.route_ids_served.str.contains('38')))]

In [9]:
ac_select = gdf[(gdf.agency == 'Alameda-Contra Costa Transit District') & ((gdf.route_ids_served.str.contains('1T')) | (gdf.route_ids_served.str.contains('72')) | (gdf.route_ids_served.str.contains('51')))]

In [10]:
vta_select = gdf[(gdf.agency == 'Santa Clara Valley Transportation Authority') & ((gdf.route_ids_served.str.contains('22')))]

In [11]:
cc_select_bus = pd.concat([muni_select, ac_select, vta_select])

In [12]:
not_just_bus = gdf[gdf.routetypes != '3']
agencies = ['Alameda-Contra Costa Transit District', 'City and County of San Francisco', 'Santa Clara Valley Transportation Authority', 'San Francisco Bay Area Rapid Transit District', 'Capitol Corridor Joint Powers Authority', 'Peninsula Corridor Joint Powers Board']
rail_origins = not_just_bus[not_just_bus.agency.isin(agencies)]

In [13]:
cc_suisun_auburn_stop_ids = ['74784', '74338', '74328', '74722', '74778', '74756', '74276']

In [14]:
rail_origins = rail_origins[~rail_origins.stop_id.isin(cc_suisun_auburn_stop_ids)]

In [15]:
# rail_origins.explore()

### CapCor Destinations

In [ ]:
agencies = ['Capitol Corridor Joint Powers Authority', 'Sacramento Regional Transit District']
rail_destinations = not_just_bus[not_just_bus.agency.isin(agencies)]
rail_destinations = rail_destinations[(rail_destinations.stop_id.isin(cc_suisun_auburn_stop_ids)) | (rail_destinations.agency == agencies[1])]

In [ ]:
# rail_destinations.explore()

### Export

#### Origins

In [ ]:
select_bus.geometry = select_bus.buffer(1000)

In [ ]:
rail_origins.geometry = rail_origins.buffer(2000)

capcor_origins = pd.concat([cc_select_bus, rail_origins])

In [ ]:
# capcor_origins.explore()

In [ ]:
export = capcor_origins[['geometry']].dissolve()

In [ ]:
export.to_file('capcor_origins_rail_alliance_2026-03-06.geojson')

In [ ]:
# export.explore()

In [ ]:
rail_destinations = rail_destinations.to_crs(CA_NAD83Albers_m)
rail_destinations.geometry = rail_destinations.buffer(2000)

In [ ]:
rail_destinations[['geometry']].dissolve().to_file('capcor_destinations_rail_alliance_2026-03-06.geojson')

## ACE/Gold Runner

* Origins
    * rail-served Bay Area 2000m buffer
    * select urban core frequent bus routes 1000m buffer (Muni 49, 38/38R, AC 1T, 72/72M/72L, 51A/51B, VTA 22/522)
    * SacRT Rail
* Destinations
    * ACE/Gold Runner valley stations 10000m (10km) buffer
* Other Constraints
    * Gateways at Antioch, Altamont Pass, Elk Grove
* Primary Insights
    * Trips viable by ACE/GR + Local Transit
* Gaps
    * Short trips

In [ ]:
not_just_bus = gdf[gdf.routetypes != '3']
agencies = ['Alameda-Contra Costa Transit District', 'City and County of San Francisco', 'Santa Clara Valley Transportation Authority', 'San Francisco Bay Area Rapid Transit District', 'Peninsula Corridor Joint Powers Board', 'Sacramento Regional Transit District']
rail_origins = not_just_bus[not_just_bus.agency.isin(agencies)]

In [ ]:
lavta = gdf[(gdf.agency == 'Livermore-Amador Valley Transit Authority')]

In [ ]:
ace_gr_select_bus = pd.concat([cc_select_bus, lavta])
ace_gr_select_bus.geometry = ace_gr_select_bus.buffer(1000)
rail_origins.geometry = rail_origins.buffer(2000)

In [ ]:
ace = not_just_bus[not_just_bus.agency == 'San Joaquin Regional Rail Commission']
ace_bay = ace[ace.stop_id.isin(ace_bay_stop_ids)]
ace_valley = ace[~ace.stop_id.isin(ace_bay_stop_ids)]

In [ ]:
ace_bay_stop_ids = ['73752', '73722', '73422', '73368', '73757', '73548', '73827']

In [ ]:
ace_bay.geometry = ace_bay.buffer(2000)

In [ ]:
gr_bay_sac_stop_ids = ['SAC', 'MTZ', 'RIC', 'BKY', 'EMY', 'OKJ', 'OAC', 'HAY', 'FMT', 'GAC', 'SCC', 'SJC']
gr_valley_stop_ids = ['LOD', 'SKT', 'SKN', 'MOD', 'TRK', 'MCD', 'MDR', 'FNO', 'HNF', 'COC', 'WAC', 'BFD']

In [ ]:
gr_bay_sac = not_just_bus[(not_just_bus.agency == 'Amtrak') & (not_just_bus.stop_id.isin(gr_bay_sac_stop_ids))]
gr_valley = not_just_bus[(not_just_bus.agency == 'Amtrak') & (not_just_bus.stop_id.isin(gr_valley_stop_ids))]

In [ ]:
gr_bay_sac.geometry = gr_bay_sac.buffer(2000)
gr_valley.geometry = gr_valley.buffer(10000)

In [ ]:
ace_gr_origins = pd.concat([rail_origins, ace_gr_select_bus, ace_bay, gr_bay_sac])

In [ ]:
# ace_gr_origins.explore()

In [ ]:
ace_gr_origins[['geometry']].dissolve().to_file('ace_gr_origins_rail_alliance_2026-03-11.geojson')

### ACE/GR Destinations

In [ ]:
ace_valley.geometry = ace_valley.buffer(10000)

In [ ]:
rail_destinations = pd.concat([ace_valley, gr_valley])

In [ ]:
# rail_destinations.explore()

In [ ]:
rail_destinations[['geometry']].dissolve().to_file('ace_gr_destinations_rail_alliance_2026-03-11.geojson')

## SoCal (Surfliner, Metrolink, NCTD)

* Spatial Constraints
    * 2000m buffer around all rail stations in region
    * 2000m buffer around bus Major Transit Stops
    * Minimum Trip 10 miles
* Primary Insights
    * Trips viable by ACE/GR + Local Transit
* Gaps
    * Short trips

In [16]:
hq_stops = catalog.ca_hq_transit_stops.read().to_crs(CA_NAD83Albers_m)

In [23]:
region = gpd.read_file('map.geojson').to_crs(CA_NAD83Albers_m)

In [25]:
gdf = hq_stops.clip(region)

In [27]:
gdf.geometry = gdf.buffer(2000)

/opt/conda/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [28]:
gdf[['geometry']].dissolve().to_file('socal_geometry_rail_alliance_2026-03-12.geojson')

## Replica Helpers

In [ ]:
def one_way_non_work(work_pct, inverse_work_pct, home_pct):
    assert work_pct < 1
    assert home_pct > inverse_work_pct and home_pct < 1
    return 1 - work_pct - inverse_work_pct

In [ ]:
one_way_non_work(.0810, .4, .561)

In [29]:
.73 * 4.96 * 10**6

3620800.0

In [30]:
.05 * .04

0.002